# Functions and Mappings

Interactive examples demonstrating functions, their properties, and ML applications.

**Topics covered:**
1. Function basics and notation
2. Domain, codomain, range
3. Injective, surjective, bijective
4. Function composition
5. Inverse functions
6. ML applications

In [ ]:
import numpy as np
np.random.seed(42)
import matplotlib.pyplot as plt

## 1. Function Basics

A **function** f: A → B maps each element in domain A to exactly one element in codomain B.

| Term | Symbol | Description |
|------|--------|-------------|
| Domain | A | Set of valid inputs |
| Codomain | B | Set of possible outputs |
| Range | f(A) | Actual outputs (⊆ codomain) |
| Image | f(x) | Output for specific input x |

In [ ]:
# Defining functions in Python

# Simple function: f(x) = x²
def square(x):
    return x ** 2

# Domain: all real numbers (in practice, what we pass)
domain = np.array([-2, -1, 0, 1, 2])
range_values = square(domain)

print(f"Domain: {domain}")
print(f"Range:  {range_values}")
print(f"\nNote: Range {set(range_values)} ⊆ Codomain (all non-negative reals)")

## 2. Function Properties: Injective, Surjective, Bijective

| Property | Definition | ML Example |
|----------|------------|------------|
| **Injective** (1-to-1) | f(a) = f(b) ⟹ a = b | Embedding layers |
| **Surjective** (onto) | ∀y∈B, ∃x: f(x)=y | Softmax (onto simplex) |
| **Bijective** | Both injective & surjective | Invertible transforms |

In [ ]:
# Testing Injectivity (one-to-one)

def is_injective(func, domain):
    """Check if function is injective on given domain."""
    outputs = [func(x) for x in domain]
    return len(outputs) == len(set(outputs))

# f(x) = x² is NOT injective on ℝ (e.g., f(-2) = f(2) = 4)
domain_real = [-2, -1, 0, 1, 2]
print(f"f(x) = x² on {domain_real}")
print(f"Injective: {is_injective(lambda x: x**2, domain_real)}")

# f(x) = x² IS injective on non-negative reals
domain_nonneg = [0, 1, 2, 3, 4]
print(f"\nf(x) = x² on {domain_nonneg}")
print(f"Injective: {is_injective(lambda x: x**2, domain_nonneg)}")

In [ ]:
# Surjectivity and Bijectivity Examples

def is_surjective(func, domain, codomain):
    """Check if function covers all of codomain."""
    range_set = set(func(x) for x in domain)
    return codomain.issubset(range_set)

# Example: Mapping students to grades
students = ['Alice', 'Bob', 'Carol', 'Dave']
grades = {'Alice': 'A', 'Bob': 'B', 'Carol': 'A', 'Dave': 'C'}
get_grade = lambda s: grades[s]

codomain = {'A', 'B', 'C', 'D', 'F'}  # All possible grades
actual_range = set(grades.values())

print(f"Grades assigned: {actual_range}")
print(f"Surjective onto {codomain}? {actual_range == codomain}")
print(f"Injective? {len(students) == len(set(grades.values()))}")  # Alice & Carol both A

## 3. Function Composition

Given f: A → B and g: B → C, the composition (g ∘ f): A → C is:

**(g ∘ f)(x) = g(f(x))**

**Key insight**: Neural networks are function compositions!
- Layer 1: f₁(x)
- Layer 2: f₂(f₁(x))
- Layer n: fₙ(fₙ₋₁(...f₁(x)))

In [ ]:
# Function Composition

def compose(g, f):
    """Return composition g ∘ f."""
    return lambda x: g(f(x))

# Example: Neural network layer = linear + activation
def linear(x, w=2, b=1):
    return w * x + b

def relu(x):
    return np.maximum(0, x)

# Single layer: relu(linear(x))
layer = compose(relu, lambda x: linear(x))

x = np.array([-2, -1, 0, 1, 2])
print(f"Input x:      {x}")
print(f"Linear(x):    {linear(x)}")
print(f"ReLU(Lin(x)): {layer(x)}")

In [ ]:
# Multi-layer Composition (Deep Network)

from functools import reduce

def make_layer(w, b):
    """Create a layer: ReLU(wx + b)."""
    return lambda x: np.maximum(0, w * x + b)

# 3-layer network
layers = [
    make_layer(2, 1),   # Layer 1
    make_layer(-1, 3),  # Layer 2  
    make_layer(0.5, 0)  # Layer 3
]

def forward(x, layers):
    """Compose all layers: fₙ ∘ ... ∘ f₁."""
    return reduce(lambda acc, f: f(acc), layers, x)

x = 1.0
print(f"Input: {x}")
for i, layer in enumerate(layers):
    x = layer(x)
    print(f"After layer {i+1}: {x}")

---

## 🔬 Deep Dive: Manual Backpropagation (Chain Rule in Action)

A neural network is **composition**: f = f₃ ∘ f₂ ∘ f₁

Backprop applies the **chain rule** layer by layer.
Here we do it **manually** and verify against numerical gradients.

```
Forward:  x → [Linear W₁] → z₁ → [ReLU] → a₁ → [Linear W₂] → y → [MSE] → loss
Backward: ∂L/∂x ← ∂L/∂y · ∂y/∂a₁ · ∂a₁/∂z₁ · ∂z₁/∂x
```

In [ ]:
# Manual Backpropagation Through a 3-Layer Network
np.random.seed(42)

# Network: x(2) → Linear(3) → ReLU → Linear(1) → MSE loss
W1 = np.random.randn(3, 2) * 0.5  # Layer 1: 2→3
b1 = np.zeros(3)
W2 = np.random.randn(1, 3) * 0.5  # Layer 2: 3→1
b2 = np.zeros(1)

x = np.array([1.0, 2.0])  # Input
y_true = np.array([1.0])   # Target

# ═══ FORWARD PASS (composition) ═══
z1 = W1 @ x + b1          # f₁: linear
a1 = np.maximum(0, z1)     # f₂: ReLU
y_pred = W2 @ a1 + b2      # f₃: linear
loss = np.mean((y_pred - y_true)**2)  # MSE

print('═══ FORWARD PASS ═══')
print(f'x     = {x}')
print(f'z₁    = W₁x + b₁ = {z1.round(4)}')
print(f'a₁    = ReLU(z₁)  = {a1.round(4)}')
print(f'ŷ     = W₂a₁ + b₂ = {y_pred.round(4)}')
print(f'loss  = MSE = {loss:.6f}')

# ═══ BACKWARD PASS (chain rule) ═══
dL_dy = 2 * (y_pred - y_true)    # ∂MSE/∂ŷ
dy_da1 = W2                       # ∂(W₂a₁+b₂)/∂a₁ = W₂
da1_dz1 = (z1 > 0).astype(float)  # ∂ReLU/∂z = 𝟙_{z>0}
dz1_dx = W1                       # ∂(W₁x+b₁)/∂x = W₁

# Chain rule: multiply through
dL_da1 = dL_dy @ dy_da1          # shape: (3,) -- error at ReLU output  
dL_dz1 = dL_da1 * da1_dz1        # element-wise * mask
dL_dx = dL_dz1 @ dz1_dx          # shape: (2,) -- error at input

print('\n═══ BACKWARD PASS (Chain Rule) ═══')
print(f'∂L/∂ŷ   = {dL_dy.round(4)}')
print(f'∂L/∂a₁  = ∂L/∂ŷ · W₂ = {dL_da1.round(4)}')
print(f'∂L/∂z₁  = ∂L/∂a₁ · 𝟙(z₁>0) = {dL_dz1.round(4)}')
print(f'∂L/∂x   = ∂L/∂z₁ · W₁ = {dL_dx.round(4)}')

# ═══ VERIFY with numerical gradient ═══
def compute_loss(x_in):
    z = W1 @ x_in + b1
    a = np.maximum(0, z)
    y = W2 @ a + b2
    return np.mean((y - y_true)**2)

eps = 1e-5
grad_num = np.array([
    (compute_loss(x + eps*np.array([1,0])) - compute_loss(x - eps*np.array([1,0]))) / (2*eps),
    (compute_loss(x + eps*np.array([0,1])) - compute_loss(x - eps*np.array([0,1]))) / (2*eps)
])

print(f'\n═══ VERIFICATION ═══')
print(f'Analytical:  {dL_dx.round(6)}')
print(f'Numerical:   {grad_num.round(6)}')
print(f'Match: {"✓" if np.allclose(dL_dx, grad_num, atol=1e-4) else "✗"}')

In [ ]:
# ══ REAL PyTorch Implementation ══
import torch
import torch.nn as nn

# Same network in PyTorch (autograd does backprop for us)
torch.manual_seed(42)

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 3)
        self.fc2 = nn.Linear(3, 1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleNet()
x = torch.tensor([1.0, 2.0], requires_grad=True)
y_true = torch.tensor([1.0])

# Forward
y_pred = model(x)
loss = nn.MSELoss()(y_pred, y_true)

# Backward — PyTorch autograd computes ALL gradients automatically
loss.backward()

print('PyTorch Autograd Backpropagation:')
print(f'  Loss:    {loss.item():.6f}')
print(f'  ∂L/∂x:  {x.grad.numpy()}')
print(f'  ∂L/∂W₁: {model.fc1.weight.grad.numpy()}')
print(f'  ∂L/∂b₁: {model.fc1.bias.grad.numpy()}')
print(f'  ∂L/∂W₂: {model.fc2.weight.grad.numpy()}')
print(f'\n→ PyTorch autograd = automatic chain rule (no manual derivatives!)')
print('→ The manual NumPy above shows WHAT autograd does under the hood')

## 4. Inverse Functions

For bijective f: A → B, the inverse f⁻¹: B → A satisfies:
- f⁻¹(f(x)) = x for all x ∈ A
- f(f⁻¹(y)) = y for all y ∈ B

**ML Applications:**
- Normalizing flows (invertible transforms)
- Encoder-decoder (approximate inverse)
- Log/exp transforms for numerical stability

In [ ]:
# Inverse Functions

# Sigmoid and its inverse (logit)
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def logit(p):
    """Inverse of sigmoid: logit(p) = log(p / (1-p))"""
    return np.log(p / (1 - p))

# Verify inverse property
x = np.array([-2, -1, 0, 1, 2])
print(f"x:               {x}")
print(f"sigmoid(x):      {sigmoid(x).round(4)}")
print(f"logit(sig(x)):   {logit(sigmoid(x)).round(4)}")
print(f"\n✓ logit(sigmoid(x)) = x")

---

## 🔬 Jacobian Matrix — Local Linear Approximation

For f: ℝⁿ → ℝᵐ, the Jacobian J captures how each output changes w.r.t. each input:

$$J_{ij} = \frac{\partial f_i}{\partial x_j}$$

- **det(J)** = how much the function stretches/compresses volume (used in normalizing flows)
- **J** = the "local linear map" that approximates f near a point

In [ ]:
# Jacobian Matrix Computation

def compute_jacobian(f, x, eps=1e-5):
    """Compute Jacobian numerically via finite differences."""
    f0 = f(x)
    n = len(x)
    m = len(f0)
    J = np.zeros((m, n))
    for j in range(n):
        x_plus = x.copy(); x_plus[j] += eps
        x_minus = x.copy(); x_minus[j] -= eps
        J[:, j] = (f(x_plus) - f(x_minus)) / (2 * eps)
    return J

# Example: f(x,y) = [x²+y, xy] at point (1, 2)
def f_example(x):
    return np.array([x[0]**2 + x[1], x[0]*x[1]])

point = np.array([1.0, 2.0])
J = compute_jacobian(f_example, point)

print(f'f({point}) = {f_example(point)}')
print(f'\nJacobian at {point}:')
print(f'  J = [[∂f₁/∂x₁, ∂f₁/∂x₂],  = [[2x, 1],  = {J.round(4)}')
print(f'       [∂f₂/∂x₁, ∂f₂/∂x₂]]    [ y, x]]')
print(f'\nAnalytical: [[{2*point[0]}, {1.0}], [{point[1]}, {point[0]}]]')
print(f'det(J) = {np.linalg.det(J):.4f}')
print(f'→ Volume scaling factor: the function stretches area by {abs(np.linalg.det(J)):.1f}x')
print(f'\n→ In normalizing flows: p_x(x) = p_z(f(x)) · |det(J)|')

# Neural network layer Jacobian
print('\n--- Neural Network Layer Jacobian ---')
W = np.array([[0.5, -0.3], [0.2, 0.8]])
def nn_layer(x): return np.maximum(0, W @ x)  # ReLU(Wx)
x0 = np.array([1.0, -0.5])
J_nn = compute_jacobian(nn_layer, x0)
print(f'Linear layer W = {W.tolist()}')
print(f'ReLU(W @ {x0}) = {nn_layer(x0)}')
print(f'Jacobian: {J_nn.round(4)}')
print(f'→ Rows with all zeros = dead neurons (ReLU killed the gradient)')

In [ ]:
# ══ REAL PyTorch Jacobian ══
import torch
from torch.autograd.functional import jacobian

# Same function f(x,y) = [x²+y, xy] — now with PyTorch autograd
def f_torch(x):
    return torch.stack([x[0]**2 + x[1], x[0]*x[1]])

point = torch.tensor([1.0, 2.0])
J = jacobian(f_torch, point)

print('PyTorch Jacobian (exact, via autograd):')
print(f'  J = {J.numpy()}')
print(f'  det(J) = {torch.linalg.det(J).item():.4f}')

# Neural network Jacobian — real use case
net = torch.nn.Sequential(
    torch.nn.Linear(3, 4),
    torch.nn.ReLU(),
    torch.nn.Linear(4, 2)
)

x = torch.randn(3)
J_nn = jacobian(lambda x: net(x), x)
print(f'\nNeural net (3→4→2) Jacobian at random input:')
print(f'  Shape: {J_nn.shape} (2 outputs × 3 inputs)')
print(f'  J = {J_nn.numpy().round(4)}')
print('\n→ torch.autograd.functional.jacobian() = exact Jacobian via autograd')
print('→ Used in normalizing flows, neural ODEs, physics-informed NNs')

## 5. ML Activation Functions

Activation functions introduce non-linearity, enabling neural networks to learn complex patterns.

| Function | Formula | Range | Use Case |
|----------|---------|-------|----------|
| Sigmoid | 1/(1+e⁻ˣ) | (0, 1) | Binary classification |
| Tanh | (eˣ-e⁻ˣ)/(eˣ+e⁻ˣ) | (-1, 1) | Hidden layers |
| ReLU | max(0, x) | [0, ∞) | Default choice |
| Softmax | eˣⁱ/Σeˣʲ | (0, 1), sum=1 | Multi-class |

In [ ]:
# Common Activation Functions

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def relu(x):
    return np.maximum(0, x)

def leaky_relu(x, alpha=0.01):
    return np.where(x > 0, x, alpha * x)

def softmax(x):
    exp_x = np.exp(x - np.max(x))  # Numerical stability
    return exp_x / exp_x.sum()

# Compare on sample input
x = np.array([-2, -1, 0, 1, 2])
print(f"Input:      {x}")
print(f"Sigmoid:    {sigmoid(x).round(3)}")
print(f"Tanh:       {tanh(x).round(3)}")
print(f"ReLU:       {relu(x)}")
print(f"LeakyReLU:  {leaky_relu(x)}")

In [ ]:
# Softmax: Maps ℝⁿ → probability simplex

logits = np.array([2.0, 1.0, 0.1])  # Raw model outputs
probs = softmax(logits)

print(f"Logits: {logits}")
print(f"Softmax output: {probs.round(4)}")
print(f"Sum of probs: {probs.sum():.4f}")
print(f"\nProperties:")
print(f"  - All positive: {(probs > 0).all()}")
print(f"  - Sum to 1: {np.isclose(probs.sum(), 1)}")
print(f"  - Surjective onto simplex ✓")

## 6. Loss Functions as Mappings

Loss functions map (predictions, targets) → scalar:
- **L**: ℝⁿ × ℝⁿ → ℝ⁺

This scalar guides optimization (minimization).

In [ ]:
# Common Loss Functions

def mse_loss(y_pred, y_true):
    """Mean Squared Error: ℝⁿ × ℝⁿ → ℝ⁺"""
    return np.mean((y_pred - y_true) ** 2)

def cross_entropy_loss(probs, y_true_idx):
    """Cross-entropy: probability × label → ℝ⁺"""
    return -np.log(probs[y_true_idx] + 1e-10)

def binary_cross_entropy(y_pred, y_true):
    """BCE: [0,1] × {0,1} → ℝ⁺"""
    return -np.mean(y_true * np.log(y_pred + 1e-10) + 
                    (1 - y_true) * np.log(1 - y_pred + 1e-10))

# Examples
y_true = np.array([1.0, 2.0, 3.0])
y_pred = np.array([1.1, 2.2, 2.8])
print(f"MSE Loss: {mse_loss(y_pred, y_true):.4f}")

probs = softmax(np.array([2.0, 1.0, 0.5]))
print(f"Cross-Entropy (class 0): {cross_entropy_loss(probs, 0):.4f}")

## 7. Feature Transformations

Common transformations for data preprocessing:

| Transform | Formula | When to Use |
|-----------|---------|-------------|
| Z-score | (x - μ) / σ | Normalize to N(0,1) |
| Min-Max | (x - min) / (max - min) | Scale to [0, 1] |
| Log | log(x) | Reduce skewness |
| Box-Cox | (xᵏ - 1) / λ | Normalize distributions |

In [ ]:
# Feature Transformations

def z_score(x):
    """Standardize to mean=0, std=1"""
    return (x - np.mean(x)) / np.std(x)

def min_max_scale(x):
    """Scale to [0, 1]"""
    return (x - np.min(x)) / (np.max(x) - np.min(x))

def log_transform(x):
    """Log transform (handles zeros)"""
    return np.log1p(x)  # log(1 + x)

# Example: Skewed data
data = np.array([1, 2, 3, 10, 100, 1000])
print(f"Original:   {data}")
print(f"Z-score:    {z_score(data).round(2)}")
print(f"Min-Max:    {min_max_scale(data).round(2)}")
print(f"Log(1+x):   {log_transform(data).round(2)}")

---

## 🔬 Embeddings: Mapping Discrete → Continuous

An **embedding** is a function f: {0, 1, ..., V-1} → ℝᵈ

It maps discrete tokens (words, pixel values, user IDs) to continuous vectors.

| Property | Embedding as Function |
|----------|----------------------|
| Domain | Finite set of token IDs |
| Codomain | ℝᵈ (d-dimensional space) |
| Injective? | Yes (each token gets unique vector) |
| Learned? | Yes — weights updated during training |

In [ ]:
# Embeddings as Functions: f(token_id) → ℝᵈ

# Simple vocabulary
vocab = {'king': 0, 'queen': 1, 'man': 2, 'woman': 3, 'prince': 4, 'princess': 5}
V = len(vocab)
d = 4  # Embedding dimension

# The embedding IS a lookup table = a function
np.random.seed(42)
# Simulate trained embeddings with semantic structure
E = np.random.randn(V, d) * 0.3
# Encode gender direction
E[0] += [0.5, 0.5, 0.0, 0.0]   # king:  royalty + male
E[1] += [0.5, -0.5, 0.0, 0.0]  # queen: royalty + female
E[2] += [-0.5, 0.5, 0.0, 0.0]  # man:   not-royalty + male
E[3] += [-0.5, -0.5, 0.0, 0.0] # woman: not-royalty + female
E[4] += [0.3, 0.3, 0.2, 0.0]   # prince
E[5] += [0.3, -0.3, 0.2, 0.0]  # princess

def embed(word):
    """f: word → ℝᵈ (the embedding function)"""
    return E[vocab[word]]

print('Embedding function outputs:')
for word in vocab:
    print(f'  f("{word}") = {embed(word).round(3)}')

# Cosine similarity: semantic closeness
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print('\nCosine similarities (semantic closeness):')
pairs = [('king','queen'), ('king','man'), ('king','woman'), ('man','woman'), ('prince','princess')]
for w1, w2 in pairs:
    sim = cosine_sim(embed(w1), embed(w2))
    bar = '█' * int(abs(sim) * 20)
    print(f'  sim("{w1}", "{w2}") = {sim:+.3f}  {bar}')

# Famous analogy: king - man + woman ≈ queen
analogy = embed('king') - embed('man') + embed('woman')
print(f'\n"king" - "man" + "woman" = ?')
for word in vocab:
    sim = cosine_sim(analogy, embed(word))
    marker = ' ← WINNER' if word == max(vocab, key=lambda w: cosine_sim(analogy, embed(w))) else ''
    print(f'  sim(result, "{word}") = {sim:.3f}{marker}')
print('→ Embedding preserves semantic relationships as geometric properties!')

In [ ]:
# ══ REAL PyTorch Embedding ══
import torch
import torch.nn as nn

# nn.Embedding = learnable lookup table: f(token_id) → ℝᵈ
vocab_size = 10000  # e.g., BPE vocabulary
embed_dim = 128     # embedding dimension

embedding = nn.Embedding(vocab_size, embed_dim)
print(f'Embedding layer: {vocab_size} tokens → ℝ^{embed_dim}')
print(f'Total parameters: {vocab_size * embed_dim:,} ({vocab_size}×{embed_dim})')

# Look up embeddings for token IDs
token_ids = torch.tensor([42, 100, 42])  # same ID → same vector!
vectors = embedding(token_ids)
print(f'\nInput:  token_ids = {token_ids.tolist()}')
print(f'Output: shape = {vectors.shape}  (3 tokens × 128 dims)')
print(f'Same ID → same vector: {torch.equal(vectors[0], vectors[2])}')

# Cosine similarity between embeddings
cos_sim = nn.CosineSimilarity(dim=0)
print(f'\ncos(token 42, token 42)  = {cos_sim(vectors[0], vectors[2]).item():.4f}')  # 1.0
print(f'cos(token 42, token 100) = {cos_sim(vectors[0], vectors[1]).item():.4f}')  # random

# How embeddings are used in a real LLM
class MiniLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_classes):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)  # f: token → ℝᵈ
        self.classifier = nn.Linear(d_model, n_classes)
    
    def forward(self, token_ids):
        x = self.embed(token_ids)       # (batch, seq_len, d_model)
        x = x.mean(dim=1)               # average pooling
        return self.classifier(x)       # (batch, n_classes)

lm = MiniLM(vocab_size=5000, d_model=64, n_classes=3)
batch = torch.randint(0, 5000, (2, 10))  # 2 docs, 10 tokens each
logits = lm(batch)
print(f'\nMini language model: {sum(p.numel() for p in lm.parameters()):,} parameters')
print(f'Input:  {batch.shape} (2 docs × 10 tokens)')
print(f'Output: {logits.shape} (2 docs × 3 classes)')
print('\n→ nn.Embedding is how EVERY LLM starts: tokens → vectors')

---

## 6. Modern Activation Functions (GELU, SwiGLU, Mish)

Modern LLMs don't use ReLU. The activation landscape has evolved:

| Activation | Formula | Used In | Key Advantage |
|-----------|---------|---------|---------------|
| ReLU | max(0,x) | Classic CNNs | Simple, fast |
| GELU | x·Φ(x) | GPT, BERT, ViT | Smooth, stochastic |
| SwiGLU | Swish(xW)⊙(xV) | LLaMA, PaLM | Gated, best for LLMs |
| Mish | x·tanh(softplus(x)) | YOLOv4 | Non-monotonic |

In [ ]:
# Modern Activation Functions Comparison
import numpy as np

def relu(x): return np.maximum(0, x)
def gelu(x): return 0.5*x*(1+np.tanh(np.sqrt(2/np.pi)*(x+0.044715*x**3)))
def swish(x): return x * (1/(1+np.exp(-x)))
def mish(x): return x * np.tanh(np.log1p(np.exp(x)))

x = np.linspace(-3, 3, 7)
print(f"{'x':>6} {'ReLU':>8} {'GELU':>8} {'Swish':>8} {'Mish':>8}")
print('-' * 42)
for xi in x:
    print(f"{xi:6.1f} {relu(xi):8.3f} {gelu(xi):8.3f} {swish(xi):8.3f} {mish(xi):8.3f}")

print('\nKey: GELU/Swish/Mish allow small negative outputs')
print('→ No dead neurons, smoother gradients')
print('→ SwiGLU = Swish + gating, used in LLaMA/PaLM/Gemma')

---

## 🔬 Gradient Flow Visualization

The most important diagnostic for deep networks: **how do gradients flow backward?**

Inspired by [UvA DL Notebooks Tutorial 3](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial3/Activation_Functions.html).

| Problem | Symptom | Cause |
|---------|---------|-------|
| Vanishing gradients | Deep layers don't learn | Sigmoid (max derivative 0.25) |
| Exploding gradients | Loss = NaN | Large weights, no normalization |
| Dead neurons | Accuracy plateaus | ReLU with negative inputs |

In [ ]:
# Gradient Flow: Sigmoid vs ReLU vs GELU across network depth
np.random.seed(0)

def simulate_gradient_flow(activation, act_deriv, n_layers=8, n_neurons=50):
    """Simulate forward+backward pass, track gradient magnitudes."""
    # He initialization
    weights = [np.random.randn(n_neurons, n_neurons) * np.sqrt(2/n_neurons) for _ in range(n_layers)]
    
    # Forward pass
    activations = [np.random.randn(n_neurons)]  # input
    pre_activations = []
    for W in weights:
        z = W @ activations[-1]
        pre_activations.append(z)
        activations.append(activation(z))
    
    # Backward pass: track gradient magnitude at each layer
    grad = np.ones(n_neurons)  # start with gradient 1
    grad_magnitudes = []
    for i in range(n_layers-1, -1, -1):
        grad = grad * act_deriv(pre_activations[i])  # element-wise
        grad = weights[i].T @ grad  # through linear layer
        grad_magnitudes.append(np.mean(np.abs(grad)))
    
    return list(reversed(grad_magnitudes))

# Activation functions and their derivatives
activations = {
    'Sigmoid': (
        lambda x: 1/(1+np.exp(-np.clip(x, -500, 500))),
        lambda x: (1/(1+np.exp(-np.clip(x,-500,500)))) * (1 - 1/(1+np.exp(-np.clip(x,-500,500))))
    ),
    'ReLU': (
        lambda x: np.maximum(0, x),
        lambda x: (x > 0).astype(float)
    ),
    'GELU': (
        lambda x: 0.5*x*(1+np.tanh(np.sqrt(2/np.pi)*(x+0.044715*x**3))),
        lambda x: np.where(np.abs(x) < 100,  # numerical stability
            0.5*(1+np.tanh(np.sqrt(2/np.pi)*(x+0.044715*x**3))) + 
            0.5*x*(1-np.tanh(np.sqrt(2/np.pi)*(x+0.044715*x**3))**2)*np.sqrt(2/np.pi)*(1+3*0.044715*x**2),
            (x > 0).astype(float))
    ),
}

print('Gradient magnitude at each layer (deeper = further from output):')
print(f'{"Layer":>7}', end='')
for name in activations: print(f'{name:>12}', end='')
print()
print('-' * 43)

results = {}
for name, (act, deriv) in activations.items():
    results[name] = simulate_gradient_flow(act, deriv)

for layer in range(8):
    print(f'{layer+1:>7}', end='')
    for name in activations:
        val = results[name][layer]
        bar = '█' * min(int(val * 20), 10)
        print(f'  {val:8.4f} {bar}', end='')
    print()

print('\n→ Sigmoid: gradients VANISH exponentially (0.25^n per layer)')
print('→ ReLU: gradients stay reasonable but some die completely')
print('→ GELU: smooth gradient flow (why Transformers use it)')

In [ ]:
# ══ REAL PyTorch Gradient Flow with Hooks ══
import torch
import torch.nn as nn

def build_net(activation, n_layers=8, width=64):
    layers = []
    for i in range(n_layers):
        layers.append(nn.Linear(width, width))
        layers.append(activation())
    layers.append(nn.Linear(width, 1))
    return nn.Sequential(*layers)

def measure_gradient_flow(model, x):
    """Use hooks to capture gradient magnitudes at every layer."""
    grad_magnitudes = {}
    hooks = []
    
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            def hook_fn(module, grad_in, grad_out, name=name):
                if grad_out[0] is not None:
                    grad_magnitudes[name] = grad_out[0].abs().mean().item()
            hooks.append(module.register_full_backward_hook(hook_fn))
    
    y = model(x)
    y.sum().backward()
    
    for h in hooks: h.remove()
    return grad_magnitudes

torch.manual_seed(42)
x = torch.randn(32, 64)  # batch of 32, 64 features

print('Real PyTorch gradient flow (using backward hooks):')
print(f'{"Layer":>8} {"Sigmoid":>10} {"ReLU":>10} {"GELU":>10}')
print('-' * 42)

for act_name, act_class in [('Sigmoid', nn.Sigmoid), ('ReLU', nn.ReLU), ('GELU', nn.GELU)]:
    torch.manual_seed(42)
    net = build_net(act_class)
    # He initialization
    for m in net.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight)
    grads = measure_gradient_flow(net, x.clone())
    if act_name == 'Sigmoid':  # store for later
        sigmoid_grads = grads
        relu_grads = None
        gelu_grads = None
    elif act_name == 'ReLU':
        relu_grads = grads
    else:
        gelu_grads = grads

# Print results
all_keys = sorted(sigmoid_grads.keys())
for key in all_keys:
    s = sigmoid_grads.get(key, 0)
    r = relu_grads.get(key, 0)
    g = gelu_grads.get(key, 0)
    print(f'{key:>8} {s:>10.6f} {r:>10.6f} {g:>10.6f}')

print('\n→ register_full_backward_hook() = how you inspect gradients in real PyTorch')
print('→ This is the exact technique used for gradient debugging in production')

---

## 🔬 Dead Neuron Detection (Dying ReLU Problem)

A **dead neuron** always outputs 0 for ALL inputs → its gradient is always 0 → it never learns.

This is the most common failure mode of ReLU networks.

In [ ]:
# Dead Neuron Detection: ReLU vs Leaky ReLU vs GELU

def count_dead_neurons(activation, n_neurons=100, n_samples=500):
    """Count neurons that output 0 for ALL inputs."""
    np.random.seed(42)
    W = np.random.randn(n_neurons, 10)  # 10-dim input
    b = np.random.randn(n_neurons) * 0.5 - 0.3  # slightly negative bias (common cause)
    
    # Test with many random inputs
    X = np.random.randn(n_samples, 10)
    outputs = np.array([activation(W @ x + b) for x in X])
    
    # A neuron is "dead" if it outputs 0 for every single input
    dead = np.all(outputs == 0, axis=0)  # check across all samples
    always_near_zero = np.all(np.abs(outputs) < 1e-6, axis=0)
    return np.sum(dead), np.sum(always_near_zero)

relu = lambda x: np.maximum(0, x)
leaky_relu = lambda x: np.where(x > 0, x, 0.01 * x)
gelu_fn = lambda x: 0.5*x*(1+np.tanh(np.sqrt(2/np.pi)*(x+0.044715*x**3)))

print('Dead neuron count (out of 100 neurons, slightly negative bias):')
print(f'  {"Activation":<15} {"Dead":>6} {"Near-zero":>10}')
print(f'  {"-"*15} {"-"*6} {"-"*10}')
for name, fn in [('ReLU', relu), ('Leaky ReLU', leaky_relu), ('GELU', gelu_fn)]:
    dead, near_zero = count_dead_neurons(fn)
    bar = '💀' * dead
    print(f'  {name:<15} {dead:>6} {near_zero:>10}  {bar}')

print('\n→ ReLU: neurons with negative bias can permanently die')
print('→ Leaky ReLU: small gradient (0.01) keeps them alive')
print('→ GELU: smooth, non-zero gradient everywhere — no dead neurons')
print('\nFixes: use GELU/Leaky ReLU, positive bias init, lower learning rate')

---

## 🔬 Weight Initialization: Starting Point in Function Space

Your **initialization** determines where you start in the function space.
Bad init → vanishing/exploding activations before training even begins.

| Init | Formula | Best For |
|------|---------|----------|
| Xavier (Glorot) | W ~ N(0, 2/(nᵢₙ+nₒᵤₜ)) | Sigmoid, Tanh |
| He (Kaiming) | W ~ N(0, 2/nᵢₙ) | ReLU, Leaky ReLU |
| Random | W ~ N(0, 1) | Nothing (too large) |

In [ ]:
# Weight Initialization: Effect on Activation Statistics
np.random.seed(42)

def forward_pass_stats(init_fn, activation, n_layers=6, width=256, input_dim=256):
    """Track mean and std of activations through layers."""
    x = np.random.randn(input_dim)  # random input
    stats = []
    for i in range(n_layers):
        W = init_fn(width, width if i > 0 else input_dim)
        x = activation(W @ x)
        stats.append((np.mean(np.abs(x)), np.std(x)))
    return stats

# Initialization schemes
random_init = lambda rows, cols: np.random.randn(rows, cols)  # too large!
xavier_init = lambda rows, cols: np.random.randn(rows, cols) * np.sqrt(2/(rows+cols))
he_init = lambda rows, cols: np.random.randn(rows, cols) * np.sqrt(2/cols)

relu_act = lambda x: np.maximum(0, x)

print('Activation statistics across 6 layers with ReLU:')
print(f'{"Layer":>5}  {"Random N(0,1)":>15}  {"Xavier":>15}  {"He (Kaiming)":>15}')
print(f'{"":>5}  {"mean|x|  std":>15}  {"mean|x|  std":>15}  {"mean|x|  std":>15}')
print('-' * 58)

stats = {
    'Random': forward_pass_stats(random_init, relu_act),
    'Xavier': forward_pass_stats(xavier_init, relu_act),
    'He': forward_pass_stats(he_init, relu_act),
}

for layer in range(6):
    print(f'{layer+1:>5}', end='')
    for name in ['Random', 'Xavier', 'He']:
        mean_abs, std = stats[name][layer]
        # Clamp for display
        mean_str = f'{mean_abs:.2e}' if mean_abs > 100 or mean_abs < 0.01 else f'{mean_abs:.4f}'
        std_str = f'{std:.2e}' if std > 100 or std < 0.01 else f'{std:.4f}'
        print(f'  {mean_str:>7} {std_str:>7}', end='')
    print()

print('\n→ Random: activations EXPLODE (values grow exponentially)')
print('→ Xavier: designed for sigmoid/tanh, shrinks with ReLU')
print('→ He: designed for ReLU, maintains stable activation magnitude')

---

## 7. Lipschitz Continuity

A function f is L-Lipschitz if: |f(x) - f(y)| ≤ L·|x - y|

This bounds how fast output can change — critical for:
- GAN stability (Wasserstein GAN requires 1-Lipschitz critic)
- Gradient clipping (bounds update size)
- Adversarial robustness (bounded perturbation effect)

In [ ]:
# Estimating Lipschitz Constants
np.random.seed(42)

def estimate_lipschitz(f, x_range=(-5,5), n=10000):
    x1 = np.random.uniform(*x_range, n)
    x2 = np.random.uniform(*x_range, n)
    return np.max(np.abs(f(x1)-f(x2)) / (np.abs(x1-x2)+1e-10))

sigmoid = lambda x: 1/(1+np.exp(-x))

print('Empirical Lipschitz constants:')
print(f'  ReLU:    {estimate_lipschitz(relu):.2f}')     # ~1.0
print(f'  Sigmoid: {estimate_lipschitz(sigmoid):.2f}')  # ~0.25
print(f'  GELU:    {estimate_lipschitz(gelu):.2f}')     # ~1.0
print(f'  x²:      {estimate_lipschitz(lambda x: x**2):.1f}')  # ~10
print(f'  tanh:    {estimate_lipschitz(np.tanh):.2f}')  # ~1.0
print('\nSigmoid L=0.25 → gradients shrink 4× per layer (vanishing!)')
print('ReLU L=1.0 → gradients preserved (why it won)')

---

## 8. Universal Approximation Theorem

The theoretical justification for neural networks:

> A feedforward network with **one hidden layer** and non-polynomial activation can
> approximate **any continuous function** to arbitrary accuracy.

**What it DOESN'T say:**
- How wide the network must be (could be huge)
- That gradient descent will find the weights
- That the result will generalize
- That one layer is efficient (depth helps immensely)

In [ ]:
# UAT Demo: Approximating sin(x) with a single hidden layer

def single_layer_network(x, W, b, v):
    """f(x) = Σ vᵢ · σ(wᵢx + bᵢ) — one hidden layer."""
    hidden = 1 / (1 + np.exp(-(W * x[:, None] + b)))  # sigmoid
    return hidden @ v

# Target function
x = np.linspace(-np.pi, np.pi, 200)
y_true = np.sin(x)

# Try different widths
np.random.seed(42)
for n_neurons in [3, 10, 50]:
    W = np.random.randn(n_neurons) * 2
    b = np.random.randn(n_neurons)
    # Least squares to find best output weights v
    H = 1 / (1 + np.exp(-(W * x[:, None] + b)))
    v = np.linalg.lstsq(H, y_true, rcond=None)[0]
    y_pred = H @ v
    mse = np.mean((y_true - y_pred)**2)
    print(f'  {n_neurons:>3} neurons: MSE = {mse:.6f}')

print('\n→ More neurons = better approximation (UAT in action)')
print('→ But 50 neurons for sin(x) is wasteful — depth is more efficient!')

---

## 🔬 Residual Connections: f(x) = x + g(x)

The **most important architectural innovation** in deep learning (ResNet, 2015).

Instead of learning f(x) directly, learn the **residual**: g(x) = f(x) - x

- Without skip: gradient must flow through ALL layers (vanishes)
- With skip: gradient has a **shortcut** through the identity path

In [ ]:
# Residual Connections: Gradient Flow Comparison
np.random.seed(42)

def gradient_flow_comparison(n_layers=10, width=30):
    """Compare gradient flow with and without residual connections."""
    weights = [np.random.randn(width, width) * np.sqrt(2/width) for _ in range(n_layers)]
    
    # Forward pass (both)
    x = np.random.randn(width)
    acts_plain, acts_resid = [x.copy()], [x.copy()]
    
    for W in weights:
        # Plain network
        acts_plain.append(np.maximum(0, W @ acts_plain[-1]))
        # Residual network: f(x) = x + ReLU(Wx) -- identity shortcut!
        acts_resid.append(acts_resid[-1] + np.maximum(0, W @ acts_resid[-1]))
    
    # Backward pass: track gradient norms
    grad_plain, grad_resid = np.ones(width), np.ones(width)
    norms_plain, norms_resid = [], []
    
    for i in range(n_layers-1, -1, -1):
        # Plain
        relu_mask = (weights[i] @ acts_plain[i] > 0).astype(float)
        grad_plain = (grad_plain * relu_mask) @ weights[i]
        norms_plain.append(np.linalg.norm(grad_plain))
        
        # Residual: ∂(x + g(x))/∂x = I + ∂g/∂x
        relu_mask_r = (weights[i] @ acts_resid[i] > 0).astype(float)
        grad_resid = grad_resid + (grad_resid * relu_mask_r) @ weights[i]  # identity + layer
        norms_resid.append(np.linalg.norm(grad_resid))
    
    return list(reversed(norms_plain)), list(reversed(norms_resid))

plain, resid = gradient_flow_comparison()

print(f'{"Layer":>5}  {"Plain ‖∇‖":>12}  {"Residual ‖∇‖":>14}')
print('-' * 35)
for i in range(10):
    p = plain[i]
    r = resid[i]
    p_bar = '█' * min(int(p/max(max(plain),1)*15), 15)
    r_bar = '█' * min(int(r/max(max(resid),1)*15), 15)
    print(f'{i+1:>5}  {p:>10.2f} {p_bar:<5}  {r:>10.2f} {r_bar}')

print(f'\nPlain:    gradient ratio (layer 1 vs 10) = {plain[0]/max(plain[-1],1e-10):.1f}x')
print(f'Residual: gradient ratio (layer 1 vs 10) = {resid[0]/max(resid[-1],1e-10):.1f}x')
print('\n→ Plain: gradients vanish in early layers (can\'t learn!)')
print('→ Residual: identity path ensures gradient ≥ 1 at every layer')
print('→ This is why ALL modern architectures use skip connections (ResNet, Transformer)')

In [ ]:
# ══ REAL PyTorch Residual Network ══
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    """f(x) = x + g(x) where g = Linear → ReLU → Linear"""
    def __init__(self, d):
        super().__init__()
        self.g = nn.Sequential(
            nn.Linear(d, d),
            nn.ReLU(),
            nn.Linear(d, d)
        )
    
    def forward(self, x):
        return x + self.g(x)  # ← THE residual connection

class PlainBlock(nn.Module):
    """f(x) = g(x) — NO skip connection"""
    def __init__(self, d):
        super().__init__()
        self.g = nn.Sequential(
            nn.Linear(d, d),
            nn.ReLU(),
            nn.Linear(d, d)
        )
    
    def forward(self, x):
        return self.g(x)  # no skip

def build_deep_net(block_class, n_blocks, d):
    blocks = [block_class(d) for _ in range(n_blocks)]
    return nn.Sequential(*blocks)

torch.manual_seed(42)
d, n_blocks = 32, 10
x = torch.randn(8, d, requires_grad=True)  # batch of 8

for name, block_cls in [('Plain (no skip)', PlainBlock), ('Residual (skip)', ResidualBlock)]:
    torch.manual_seed(42)
    net = build_deep_net(block_cls, n_blocks, d)
    
    # Gradient stats per block
    y = net(x)
    y.sum().backward()
    
    grad_norm = x.grad.norm().item()
    print(f'{name:>20}: ‖∂L/∂x‖ = {grad_norm:.6f}')
    x.grad = None  # reset

print('\n→ Plain: gradient through 10 blocks → vanishes to near-zero')
print('→ Residual: identity path guarantees gradient flow')
print('→ ResNet, Transformer, GPT ALL use residual connections')

---

## 🔬 Attention: Adaptive Function Composition

An MLP applies the **same** function to every input: f(x) = σ(Wx + b).

**Attention** computes a **different** function for each input — it's adaptive composition.

| Architecture | Composition Type | Fixed? |
|-------------|-----------------|--------|
| MLP | f₃ ∘ f₂ ∘ f₁ | Same for every x |
| Attention | Σᵢ αᵢ(x) · fᵢ(x) | Changes per input |
| Mixture of Experts | f_{router(x)}(x) | Selects expert per input |

In [ ]:
# Attention = Input-Dependent Function Composition

# Tokens: ["The", "cat", "sat", "on", "mat"]
np.random.seed(42)
seq_len, d_model = 5, 8
tokens = ['The', 'cat', 'sat', 'on', 'mat']

# Random token embeddings
X = np.random.randn(seq_len, d_model)

# Learned projections (simplified single-head attention)
W_Q = np.random.randn(d_model, d_model) * 0.3  # Query
W_K = np.random.randn(d_model, d_model) * 0.3  # Key
W_V = np.random.randn(d_model, d_model) * 0.3  # Value

Q = X @ W_Q  # What am I looking for?
K = X @ W_K  # What do I contain?
V = X @ W_V  # What do I offer?

# Attention scores: how much each token attends to each other
scores = Q @ K.T / np.sqrt(d_model)
# Softmax (fuzzy membership!)
def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

attn_weights = softmax(scores)

print('Attention weights (each row = one token\'s "function composition"):')
print(f'{"":>6}', end='')
for t in tokens: print(f'{t:>6}', end='')
print()
for i, tok in enumerate(tokens):
    print(f'{tok:>6}', end='')
    for j in range(seq_len):
        w = attn_weights[i, j]
        bar = '█' * int(w * 10)
        print(f' {w:.2f}{bar}', end='')
    print()

# Output = weighted combination (adaptive composition)
output = attn_weights @ V

print('\n→ Each token computes a DIFFERENT weighted sum of value vectors')
print('→ MLP: every input gets the SAME transformation Wx+b')
print('→ Attention: each input gets its OWN custom transformation')
print('→ This is why Transformers are so powerful — adaptive function composition!')

In [ ]:
# ══ REAL PyTorch Self-Attention ══
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SelfAttention(nn.Module):
    """Single-head self-attention (the core of every Transformer)."""
    def __init__(self, d_model):
        super().__init__()
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.d_model = d_model
    
    def forward(self, x, mask=None):
        Q = self.W_Q(x)  # (batch, seq, d)
        K = self.W_K(x)
        V = self.W_V(x)
        
        # Scaled dot-product attention
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_model)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)
        return attn_weights @ V, attn_weights

# Use it
torch.manual_seed(42)
d_model = 16
seq_len = 5
attn = SelfAttention(d_model)

# Simulated token embeddings: [The, cat, sat, on, mat]
tokens = ['The', 'cat', 'sat', 'on', 'mat']
x = torch.randn(1, seq_len, d_model)  # (batch=1, seq=5, d=16)

output, weights = attn(x)
print(f'Input:  {x.shape}  → (1 batch × {seq_len} tokens × {d_model} dims)')
print(f'Output: {output.shape} → same shape (attention doesn\'t change dimensions)')

print(f'\nAttention weights (which tokens attend to which):')
w = weights[0].detach().numpy()
print(f'{"":>6}', end='')
for t in tokens: print(f'{t:>6}', end='')
print()
for i, tok in enumerate(tokens):
    print(f'{tok:>6}', end='')
    for j in range(seq_len):
        print(f'{w[i,j]:6.2f}', end='')
    print()

# Causal mask (for autoregressive LLMs like GPT)
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)
output_causal, weights_causal = attn(x, mask=causal_mask)

print(f'\nWith causal mask (GPT-style, can only attend to past):')
wc = weights_causal[0].detach().numpy()
print(f'{"":>6}', end='')
for t in tokens: print(f'{t:>6}', end='')
print()
for i, tok in enumerate(tokens):
    print(f'{tok:>6}', end='')
    for j in range(seq_len):
        print(f'{wc[i,j]:6.2f}', end='')
    print()

print(f'\nTotal parameters: {sum(p.numel() for p in attn.parameters())}')
print('→ This is the EXACT attention mechanism inside GPT, BERT, LLaMA')
print('→ F.softmax + scaled dot-product = how every Transformer works')

---

## 9. Function Spaces — Choosing Your Model

When you pick a model architecture, you're choosing a **function space** (hypothesis class H):

| Architecture | Function Space | Capacity |
|-------------|---------------|----------|
| Linear regression | H = {f: f(x) = w·x + b} | Very small |
| 2-layer MLP (100 units) | H ⊂ continuous functions | Moderate |
| ResNet-50 | Huge H with skip connections | Large |
| GPT-3 (175B params) | Enormous H over sequences | Very large |

**Bias-variance**: Small H → underfitting, Large H → overfitting. Regularization restricts H.

## Summary

### Core Function Properties:
| Property | Definition | ML Connection |
|----------|-----------|---------------|
| Injective | f(a)=f(b)⟹a=b | Invertible layers |
| Surjective | Range = Codomain | Full coverage |
| Bijective | Both | Normalizing flows |
| Lipschitz | ‖f(x)-f(y)‖ ≤ L‖x-y‖ | GAN stability |
| Composition | (g∘f)(x) = g(f(x)) | Deep networks = composition |

### Key Concepts Added:
- **GELU/SwiGLU** replace ReLU in modern LLMs
- **UAT**: one hidden layer CAN approximate anything (but depth is efficient)
- **Lipschitz**: bounds output change → gradient stability
- **Function spaces**: architecture ≡ hypothesis class